# Step 3 - Re-check the candidates with Stockfish and build the final dataset

**Goal:** Lichess evaluations are only a pre-filter. Here every candidate is analysed with *your own* Stockfish
(fixed settings -> reproducible) and kept only if Stockfish confirms the same label (good / bad).

**Input:** `outputs/step2_select_good_bad_moves/candidates.jsonl`
**Output** (`outputs/step3_stockfish_check/`): `analysed_all.jsonl` (every candidate + reject reason),
`positions.jsonl` / `positions.csv` (the **final dataset**), `kpis.json`, `manifest.json`

**Stockfish settings** (from `config.py`): fixed depth 18, 1 thread, 128 MB hash, MultiPV 1, hash cleared before each
search (a new `game` tag per call makes python-chess send `ucinewgame`). Fixed depth + one thread = deterministic.

For each candidate we analyse the position *before* the move (-> engine best move, evaluation before) and *after* the
human move (-> evaluation after and the principal variation, needed later for Condition C).
Evaluations are stored from White's point of view; `cp_loss` is from the mover's point of view.
The file is resumable: if it stops, run it again and finished candidates are skipped.

In [ ]:
import json, shutil, sys
from pathlib import Path
import chess, chess.engine
import pandas as pd
import config as C
from move_rules import classify
from common import sha256_file, write_manifest

IN = C.OUT_DIR / "step2_select_good_bad_moves" / "candidates.jsonl"
OUT = C.OUT_DIR / "step3_stockfish_check"
OUT.mkdir(parents=True, exist_ok=True)

def find_stockfish():
    p = Path(C.STOCKFISH_PATH)
    p = p if p.is_absolute() else C.ROOT / p
    if p.exists():
        return str(p)
    found = shutil.which("stockfish")
    if found:
        return found
    sys.exit(f"Stockfish not found. Set STOCKFISH_PATH (tried {p})")

candidates = [json.loads(l) for l in open(IN, encoding="utf-8")]
print(len(candidates), "candidates")

## Analysis loop (resumable)

In [ ]:
def analyse(engine, board, tag):
    info = engine.analyse(board, chess.engine.Limit(depth=C.STOCKFISH_DEPTH), multipv=C.STOCKFISH_MULTIPV, game=tag)
    return info[0] if isinstance(info, list) else info

def pv_san(board, pv, n):
    b, out = board.copy(), []
    for mv in pv[:n]:
        out.append(b.san(mv)); b.push(mv)
    return out

all_path = OUT / "analysed_all.jsonl"
done = {json.loads(l)["cand_id"] for l in open(all_path, encoding="utf-8")} if all_path.exists() else set()
sf = find_stockfish()
engine = chess.engine.SimpleEngine.popen_uci(sf)
engine.configure({"Threads": C.STOCKFISH_THREADS, "Hash": C.STOCKFISH_HASH_MB})
engine_name = engine.id.get("name", "unknown")
print("Engine:", engine_name)

try:
    with open(all_path, "a", encoding="utf-8") as fo:
        for c in candidates:
            if c["cand_id"] in done:
                continue
            board = chess.Board(c["fen_before"]); mover = board.turn
            move = chess.Move.from_uci(c["move_uci"])
            before = analyse(engine, board, c["cand_id"] + "_before")
            after_board = board.copy(); after_board.push(move)
            after = analyse(engine, after_board, c["cand_id"] + "_after")
            best = before["pv"][0]
            row = dict(c, engine_best_uci=best.uci(), engine_best_san=board.san(best),
                       engine_best_pv_san=pv_san(board, before["pv"], C.PV_MAX_PLIES),
                       human_move_is_engine_best=(best == move), rejected_reasons=[])
            if before["score"].is_mate() or after["score"].is_mate():
                row["rejected_reasons"].append("mate_score")
            else:
                label, cp_loss, wp_loss = classify(before["score"].pov(mover).score(), after["score"].pov(mover).score())
                row.update(eval_before_cp=before["score"].white().score(), eval_after_cp=after["score"].white().score(),
                           cp_loss=cp_loss, wp_loss=round(wp_loss, 1), stockfish_label=label,
                           pv_after_san=pv_san(after_board, after.get("pv", []), C.PV_MAX_PLIES),
                           depth_reached=after.get("depth"))
                if abs(row["eval_before_cp"]) > C.MAX_ABS_EVAL_CP:
                    row["rejected_reasons"].append("already_decided")
                if label != c["label"]:
                    row["rejected_reasons"].append("label_not_confirmed_by_stockfish")
            fo.write(json.dumps(row) + "\n"); fo.flush()
finally:
    engine.quit()

## Final dataset
Kept = no reject reason. Positions are numbered `P001...` in file order. (Reminder from your professor: keep the pilot
positions out of the final dataset - after the pilot, delete those games here or exclude their `game_id`s.)

In [ ]:
EXCLUDE_GAME_IDS = set()      # fill with the pilot's game ids
analysed = [json.loads(l) for l in open(all_path, encoding="utf-8")]
kept = [r for r in analysed if not r["rejected_reasons"] and r["game_id"] not in EXCLUDE_GAME_IDS]
for i, r in enumerate(kept, 1):
    r["pos_id"] = f"P{i:03d}"
with open(OUT / "positions.jsonl", "w", encoding="utf-8") as f:
    for r in kept:
        f.write(json.dumps(r) + "\n")
df = pd.DataFrame(kept)
df.drop(columns=["rejected_reasons"]).to_csv(OUT / "positions.csv", index=False)
rej = pd.Series([x for r in analysed for x in r["rejected_reasons"]]).value_counts()
print(f"{len(analysed)} analysed -> {len(kept)} kept")
print("\nRejected, by reason:\n", rej.to_string() if len(rej) else "none")

## KPIs of the final positions (+ duplicate check)
These are the numbers your email says are still to be added.

In [ ]:
fen_key = df["fen_before"].str.split().str[:4].str.join(" ")
kpis = dict(
    n_positions=len(df), n_games=int(df["game_id"].nunique()),
    by_label=df["label"].value_counts().to_dict(),
    duplicate_positions=int(fen_key.duplicated().sum()), duplicate_candidate_ids=int(df["cand_id"].duplicated().sum()),
    games_with_both_labels=int((df.groupby("game_id")["label"].nunique() == 2).sum()),
    human_move_is_engine_best_share=round(float(df["human_move_is_engine_best"].mean()), 3),
    white_to_move_share=round(float(df["fen_before"].str.contains(" w ").mean()), 3),
)
display_cols = ["cp_loss", "wp_loss", "eval_before_cp", "fullmove", "depth_reached"]
print(json.dumps(kpis, indent=2))
(OUT / "kpis.json").write_text(json.dumps(kpis, indent=2))
df.groupby("label")[display_cols].describe().T

In [ ]:
write_manifest(OUT / "manifest.json", "step3_stockfish_check",
               {"engine_name": engine_name, "engine_sha256": sha256_file(sf), "input_sha256": sha256_file(IN),
                "output_sha256": sha256_file(OUT / "positions.jsonl"), "kpis": kpis})
print("Manifest written. Final dataset:", OUT / "positions.csv")